# Fase 3 - Regressao Polinomial

Julia Welter, Pedro Pacheco, Pedro K. Albuquerque e Tiago Rosa 

Notebook completo e autossuficiente para entrega da fase 3. Todo o codigo necessario esta neste arquivo.

## Bibliotecas e Funcoes Auxiliares

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io as scipy

try:
    from sklearn.metrics import mean_squared_error, r2_score
    from sklearn.model_selection import train_test_split
except ImportError:
    mean_squared_error = None
    r2_score = None
    train_test_split = None

GRAUS = [1, 2, 3, 8]
CORES = {1: 'r', 2: 'g', 3: 'k', 8: 'y', 20: 'm'}

plt.rcParams['figure.figsize'] = (10, 6)

def carregar_dados():
    file_mat = 'dados/data_preg.mat'
    file_csv = 'dados/data_preg.csv'

    if os.path.exists(file_mat):
        mat = scipy.loadmat(file_mat)
        if 'data' in mat:
            data = mat['data']
        else:
            keys = [k for k in mat.keys() if not k.startswith('__')]
            data = mat[keys[0]]
        return data[:, 0], data[:, 1]

    data = pd.read_csv(file_csv, header=None)
    return data[0].values, data[1].values

def calcular_polinomio(coeffs, x_vals):
    n = len(coeffs) - 1
    y_vals = np.zeros_like(x_vals, dtype=float)
    for i, beta in enumerate(coeffs):
        y_vals += beta * (x_vals ** (n - i))
    return y_vals

def dividir_treino_teste(x, y, test_size=0.3, random_state=42):
    if train_test_split is not None:
        return train_test_split(x, y, test_size=test_size, random_state=random_state)

    rng = np.random.default_rng(random_state)
    indices = np.arange(len(x))
    rng.shuffle(indices)
    qtd_teste = int(np.ceil(len(indices) * test_size))
    indices_teste = indices[:qtd_teste]
    indices_treino = indices[qtd_teste:]

    return (
        x[indices_treino],
        x[indices_teste],
        y[indices_treino],
        y[indices_teste],
    )

def calcular_mse(y_real, y_prev):
    if mean_squared_error is not None:
        return mean_squared_error(y_real, y_prev)
    return float(np.mean((y_real - y_prev) ** 2))

def calcular_r2(y_real, y_prev):
    if r2_score is not None:
        return r2_score(y_real, y_prev)

    soma_residuos = np.sum((y_real - y_prev) ** 2)
    soma_total = np.sum((y_real - np.mean(y_real)) ** 2)
    return float(1 - (soma_residuos / soma_total))

def criar_grafico_base(x, y):
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(x, y, color='blue', label='Dados Originais', alpha=0.6)
    return fig, ax

def adicionar_curva(ax, x, y_ajustado, grau):
    ordem = np.argsort(x)
    ax.plot(x[ordem], y_ajustado[ordem], CORES[grau], label=f'Regressao N={grau}')

def finalizar_grafico(ax, titulo='Regressao Polinomial - Fase 3'):
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(titulo)
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.7)
    plt.show()


## Questao A

Carregamento dos dados do arquivo `.mat` ou `.csv`.

In [ ]:
x, y = carregar_dados()
print(f'Total de amostras: {len(x)}')
print(f'Intervalo de x: {x.min():.4f} ate {x.max():.4f}')
print(f'Intervalo de y: {y.min():.4f} ate {y.max():.4f}')
pd.DataFrame({'x': x, 'y': y}).head()


## Questao B

Grafico de dispersao dos dados originais.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(x, y, color='blue', label='Dados Originais', alpha=0.6)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Regressao Polinomial - Fase 3')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()


## Questoes C, D, E e F

Ajuste e exibicao de modelos polinomiais de graus 1, 2, 3 e 8 no mesmo grafico.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(x, y, color='blue', label='Dados Originais', alpha=0.6)

for grau in GRAUS:
    beta = np.polyfit(x, y, grau)
    y_ajustado = calcular_polinomio(beta, x)
    ordem = np.argsort(x)
    plt.plot(x[ordem], y_ajustado[ordem], CORES[grau], label=f'Regressao N={grau}')

plt.xlabel('x')
plt.ylabel('y')
plt.title('Regressao Polinomial - Fase 3')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()


## Questao G

Divisao treino/teste e calculo das metricas MSE e R2 para os graus 1, 2, 3 e 8.

In [ ]:
x_treino_g, x_teste_g, y_treino_g, y_teste_g = dividir_treino_teste(x, y)
x_base = np.linspace(x.min(), x.max(), 300)
resultados_g = []

for grau in GRAUS:
    coeficientes = np.polyfit(x_treino_g, y_treino_g, grau)
    y_prev_treino = calcular_polinomio(coeficientes, x_treino_g)
    y_prev_teste = calcular_polinomio(coeficientes, x_teste_g)
    y_base = calcular_polinomio(coeficientes, x_base)
    resultados_g.append({
        'grau': grau,
        'coeficientes': coeficientes,
        'mse_treino': calcular_mse(y_treino_g, y_prev_treino),
        'mse_teste': calcular_mse(y_teste_g, y_prev_teste),
        'r2_treino': calcular_r2(y_treino_g, y_prev_treino),
        'r2_teste': calcular_r2(y_teste_g, y_prev_teste),
        'y_base': y_base,
    })

print('\n' + '=' * 72)
print(f"{'Grau':>6}  {'MSE Treino':>14}  {'MSE Teste':>14}  {'R2 Treino':>10}  {'R2 Teste':>10}")
print('-' * 72)

melhor_g = min(resultados_g, key=lambda r: r['mse_teste'])

for r in resultados_g:
    marca = ' <- ' if r['grau'] == melhor_g['grau'] else '  '
    print(
        f"{r['grau']:>6}  {r['mse_treino']:>14.4f}  {r['mse_teste']:>14.4f}  {r['r2_treino']:>10.4f}  {r['r2_teste']:>10.4f}{marca}"
    )

print('=' * 72)
print(f"\nModelo mais preciso (menor MSE no teste): Grau {melhor_g['grau']}")
print(f"MSE Teste = {melhor_g['mse_teste']:.4f}  |  R2 Teste = {melhor_g['r2_teste']:.4f}")

print(f"\nCoeficientes do grau {melhor_g['grau']}:")
n = melhor_g['grau']
for i, beta in enumerate(melhor_g['coeficientes']):
    exp = n - i
    nome = f'Beta{exp}' if exp > 0 else 'Beta0'
    print(f"{nome:>6} = {beta:+.6f}")

fig, ax = criar_grafico_base(x, y)
for r in resultados_g:
    adicionar_curva(ax, x_base, r['y_base'], r['grau'])
finalizar_grafico(ax)


## Questao H

Separacao dos dados em 90% para treinamento e 10% para teste.

In [ ]:
x_treino, x_teste, y_treino, y_teste = dividir_treino_teste(x, y, test_size=0.1)
print('--- Questao H ---')
print(f'Dados totais: {len(x)}')
print(f'Dados de treinamento: {len(x_treino)}')
print(f'Dados de teste: {len(x_teste)}')


## Questao I

Ajuste polinomial usando apenas os dados de treinamento.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(x_treino, y_treino, color='blue', label='Treino (90%)', alpha=0.5)
plt.scatter(x_teste, y_teste, color='orange', label='Teste (10%)', marker='x')

x_min = min(x_treino.min(), x_teste.min())
x_max = max(x_treino.max(), x_teste.max())
x_suave = np.linspace(x_min, x_max, 500)

print('Ajustando modelos apenas com dados de treinamento...')

for grau in GRAUS:
    beta = np.polyfit(x_treino, y_treino, grau)
    y_curva = calcular_polinomio(beta, x_suave)
    plt.plot(x_suave, y_curva, CORES[grau], label=f'Regressao N={grau} (Treino)')

plt.xlabel('x')
plt.ylabel('y')
plt.title('Questao I: Ajuste Polinomial apenas com Dados de Treino')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()


## Questao J

Comparacao dos modelos com foco apenas no desempenho do conjunto de teste.

In [ ]:
x_treino_j, x_teste_j, y_treino_j, y_teste_j = dividir_treino_teste(x, y)
x_base_j = np.linspace(x.min(), x.max(), 300)
resultados_j = []

for grau in GRAUS:
    coeficientes = np.polyfit(x_treino_j, y_treino_j, grau)
    y_prev_teste = calcular_polinomio(coeficientes, x_teste_j)
    y_base = calcular_polinomio(coeficientes, x_base_j)
    resultados_j.append({
        'grau': grau,
        'coeficientes': coeficientes,
        'mse_teste': calcular_mse(y_teste_j, y_prev_teste),
        'r2_teste': calcular_r2(y_teste_j, y_prev_teste),
        'y_base': y_base,
    })

melhor_j = min(resultados_j, key=lambda r: r['mse_teste'])

print('\n' + '=' * 50)
print(f"{'Grau':>6}  {'MSE Teste':>14}  {'R2 Teste':>10}")
print('-' * 50)
for r in resultados_j:
    marca = ' <- Melhor' if r['grau'] == melhor_j['grau'] else ''
    print(f"{r['grau']:>6}  {r['mse_teste']:>14.4f}  {r['r2_teste']:>10.4f}{marca}")
print('=' * 50)
print(f"\nMais preciso (menor MSE no teste): Grau {melhor_j['grau']}")
print(f"MSE = {melhor_j['mse_teste']:.4f}  |  R2 = {melhor_j['r2_teste']:.4f}")

fig, ax = criar_grafico_base(x, y)
for r in resultados_j:
    adicionar_curva(ax, x_base_j, r['y_base'], r['grau'])
finalizar_grafico(ax)


## Questao K

Calculo do R2 para treino e teste de cada modelo.

In [ ]:
x_treino_k, x_teste_k, y_treino_k, y_teste_k = dividir_treino_teste(x, y)
resultados_k = []

for grau in GRAUS:
    coeficientes = np.polyfit(x_treino_k, y_treino_k, grau)
    y_prev_treino = calcular_polinomio(coeficientes, x_treino_k)
    y_prev_teste = calcular_polinomio(coeficientes, x_teste_k)
    resultados_k.append({
        'grau': grau,
        'mse_treino': calcular_mse(y_treino_k, y_prev_treino),
        'mse_teste': calcular_mse(y_teste_k, y_prev_teste),
        'r2_treino': calcular_r2(y_treino_k, y_prev_treino),
        'r2_teste': calcular_r2(y_teste_k, y_prev_teste),
    })

print('k) Calculo do R2 para treino e teste:')
for resultado in resultados_k:
    print(
        f"N={resultado['grau']}: MSE treino={resultado['mse_treino']:.6f}, MSE teste={resultado['mse_teste']:.6f}, R2 treino={resultado['r2_treino']:.6f}, R2 teste={resultado['r2_teste']:.6f}"
    )

melhor_r2 = max(resultados_k, key=lambda item: item['r2_teste'])
print('Conclusao: quanto mais proximo de 1 estiver o R2, melhor o modelo explica a variacao dos dados. Se o R2 de treino for muito alto e o de teste cair, isso indica overfitting.')
print(f"Neste experimento, o maior R2 de teste foi no modelo N={melhor_r2['grau']} com R2 teste={melhor_r2['r2_teste']:.6f}.")


## Questao L

Definicao do modelo mais preciso com base no menor erro no conjunto de teste.

In [ ]:
melhor_modelo = min(resultados_k, key=lambda item: item['mse_teste'])
print('l) Modelo mais preciso:')
print(
    f"O modelo mais preciso neste caso e o de grau N={melhor_modelo['grau']}, pois apresentou o menor erro no conjunto de teste (MSE teste={melhor_modelo['mse_teste']:.6f}) e o melhor poder de generalizacao (R2 teste={melhor_modelo['r2_teste']:.6f})."
)
melhor_modelo


## Questao M

Inclusao do modelo de grau 20 para comparar desempenho e discutir overfitting.

In [ ]:
graus_m = [1, 2, 3, 8, 20]
resultados_m = []

plt.figure(figsize=(12, 7))
plt.scatter(x_treino, y_treino, color='blue', alpha=0.4, label='Treino')
plt.scatter(x_teste, y_teste, color='orange', label='Teste', marker='x')

x_suave_m = np.linspace(x_treino.min(), x_treino.max(), 1000)

print(f"{'Grau':<6} | {'MSE Treino':<12} | {'MSE Teste':<12}")
print('-' * 40)

for grau in graus_m:
    beta = np.polyfit(x_treino, y_treino, grau)
    y_pred_tr = calcular_polinomio(beta, x_treino)
    y_pred_te = calcular_polinomio(beta, x_teste)
    mse_tr = calcular_mse(y_treino, y_pred_tr)
    mse_te = calcular_mse(y_teste, y_pred_te)
    resultados_m.append({'grau': grau, 'mse_treino': mse_tr, 'mse_teste': mse_te})
    print(f"{grau:<6} | {mse_tr:<12.5f} | {mse_te:<12.5f}")

    y_plot = calcular_polinomio(beta, x_suave_m)
    plt.plot(x_suave_m, y_plot, CORES[grau], label=f'N={grau}')

plt.ylim(y_treino.min() - 2, y_treino.max() + 2)
plt.title('Comparacao de Modelos e Overfitting (N=20)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Justificativa da Questao M

O modelo de grau 20 apresenta pior desempenho no teste porque sofre de overfitting.

Apesar de ter o menor erro no treino, ele memoriza o ruido dos dados em vez de aprender o padrao real, resultando em uma curva extremamente oscilatoria e cheia de zigue-zagues.

Essa complexidade excessiva faz com que o modelo perca a capacidade de generalizacao, tornando-o incapaz de prever com precisao novos dados.